# Homework 1 — Scientific Paper Processing with spaCy

Today's first experiment: extract text from PDF pages while preserving page boundaries.

We will inspect the output on the first, middle, and last pages, then record errors in `notes/experiment_log.md`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from scipaper.extraction import extract_pdf, document_statistics, save_extracted_text

## PDF paths

The course PDFs are expected in the repository-local `data/papers/` directory.


In [ ]:
PAPER_DIR = PROJECT_ROOT / 'data' / 'papers'
PDF_FILES = [
    PAPER_DIR / 'AttentionYouNeed.pdf',
    PAPER_DIR / '1406.1078v3.pdf',
    PAPER_DIR / '1409.0473v7.pdf',
    PAPER_DIR / '1412.3555v1.pdf',
    PAPER_DIR / '1601.06733v7.pdf',
    PAPER_DIR / '1607.06450v1.pdf',
    PAPER_DIR / '1610.02357v3.pdf',
    PAPER_DIR / '1703.03906v2.pdf',
    PAPER_DIR / 'N16-1024.pdf',
]
SORT_BLOCKS = True  # Initial run; compare with False on the same pages.
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'extracted_text'

missing = [str(path) for path in PDF_FILES if not path.is_file()]
if missing:
    print('PDF paths not found:', missing)
else:
    print(f'{len(PDF_FILES)} PDFs ready from {PAPER_DIR}')

In [ ]:
results = []
for pdf_path in PDF_FILES:
    pages = extract_pdf(pdf_path, sort_blocks=SORT_BLOCKS)
    text_path = OUTPUT_DIR / f'{pdf_path.stem}.txt'
    save_extracted_text(pages, text_path)
    results.append({'paper': pdf_path.stem, **document_statistics(pages), 'text_file': str(text_path)})

results

In [ ]:
# Inspect short excerpts from the first, middle, and last pages.
for pdf_path in PDF_FILES:
    pages = extract_pdf(pdf_path, sort_blocks=SORT_BLOCKS)
    selected = sorted({0, len(pages) // 2, len(pages) - 1})
    print(f'\n### {pdf_path.name}')
    for index in selected:
        print(f'\n--- page {pages[index]["page"]} ---\n{pages[index]["text"][:1200]}')

## Task 2 — Section and subsection detection

This experiment detects headings from PDF layout blocks. The first rule uses numbered or lettered section IDs together with font emphasis or a size increase. A second rule handles compact small-caps headings whose PDFs encode the text in a regular font. Common unnumbered labels such as Abstract, Acknowledgments, and References are also included. The detector records the one-based PDF page and does not contain paper-specific section lists.

The results are candidates for review, not guaranteed ground truth. We manually inspected representative pages from different layouts and recorded false positives, missed headings, and limitations in `notes/experiment_log.md`. The sample is too small to claim perfect recall or precision across every page.

In [ ]:
from scipaper.sections import detect_sections

section_results = {
    pdf_path.stem: detect_sections(pdf_path)
    for pdf_path in PDF_FILES
}

for paper_id, sections in section_results.items():
    print(f"{paper_id}: {len(sections)} heading candidates")
    for section in sections:
        section_id = section["section_id"] or "(unnumbered)"
        print(f"  page {section['page']:>2} | {section_id:<7} | {section['title']}")
    print()

### Manual review checkpoint

Useful pages to compare against the rendered PDFs include `1409.0473v7` page 2 (small-caps subsection 2.1), `1610.02357v3` page 2 (wrapped subsection 1.2), `1703.03906v2` page 3 (two-column subsections 3.2 and 3.3), and `N16-1024` page 2 (sections 2, 3, and 3.1). Review the visual source before treating any candidate as correct; the experiment log records what was checked and what remains uncertain.

## Task 3 — Paragraph detection and manual evaluation

The manually reviewed pages are now a small development reference set. The layout-block baseline performed better on the reviewed two-column prose page: it kept 10 body paragraphs separate, while blank-line splitting merged the page into one candidate. Both methods preserved the four paragraphs on the reviewed Transformer figure page. On the table/equation page, we confirmed which blocks are not paragraphs and observed that layout blocks split the two equation lines. These observations motivate a provisional layout-block pipeline with explicit exclusions and an audit trail; they do not establish corpus-wide accuracy.


The latest revision also excludes first-page title/author blocks before the first detected structural heading as `front_matter`, while retaining the abstract body. Current batch: 1,396 paragraph records across 9 papers (107 pages). The two complete manual pages still match 14/14 paragraphs exactly; this is development evidence only, not corpus-wide accuracy. See the latest experiment-log entry for updated per-paper counts and limits.

In [ ]:
from scipaper.paragraphs import extract_paragraph_candidates

PARAGRAPH_REVIEW_PAGES = [
    ("1406.1078v3.pdf", 5),       # two-column prose; manually labeled
    ("AttentionYouNeed.pdf", 3),  # figure, headings, and prose; manually labeled
    ("AttentionYouNeed.pdf", 6),  # table and displayed equation group
]

for paper_name, pdf_page in PARAGRAPH_REVIEW_PAGES:
    pdf_path = PAPER_DIR / paper_name
    blank = extract_paragraph_candidates(
        pdf_path, method="blank_lines", pages={pdf_page}
    )
    blocks = extract_paragraph_candidates(
        pdf_path, method="layout_blocks", pages={pdf_page}
    )
    print(
        f"{paper_name}, PDF page {pdf_page}: "
        f"blank-line candidates={len(blank)}, "
        f"layout-block candidates={len(blocks)}"
    )


### Paragraph records for all supplied papers

The production baseline uses layout blocks, then applies general rules for detected headings, captions, table regions with visible rules, equations, page numbers, and short isolated fragments. It assigns each retained paragraph its PDF page, current detected section, paper ID, paragraph number, and source candidate IDs. Inspect `excluded` as well as `paragraphs`: exclusions are heuristic and are kept visible so we can spot mistakes. The manual score below uses only fully reviewed pages; the table/equation page is a partial review and is not included in precision/recall.


In [ ]:
import json

from scipaper.paragraphs import extract_document_paragraphs

paragraph_results = {
    pdf_path.stem: extract_document_paragraphs(pdf_path)
    for pdf_path in PDF_FILES
}

for paper_id, result in paragraph_results.items():
    stats = result["statistics"]
    print(
        f"{paper_id}: {stats['paragraphs']} paragraphs, "
        f"{stats['detected_sections']} detected sections, "
        f"{stats['excluded_candidates']} excluded candidates"
    )


### Exact paragraph match on the manually reviewed pages

A paragraph is an exact match only when the pipeline retains the same source-block group as the manual paragraph label. This score describes these reviewed development examples only; it is not an estimate of accuracy across all pages.


In [ ]:
review_path = PROJECT_ROOT / "data" / "annotations" / "task3_manual_review.json"
manual_reviews = json.loads(review_path.read_text(encoding="utf-8"))["reviews"]

for review in manual_reviews:
    if not review["complete_page_review"]:
        continue
    result = paragraph_results[review["paper"]]
    gold_segments = {
        frozenset({int(candidate_id)})
        for candidate_id, label in review["labels"].items()
        if label == "paragraph"
    }
    predicted_segments = {
        frozenset(paragraph["source_candidates"])
        for paragraph in result["paragraphs"]
        if paragraph["page"] == review["page"]
    }
    true_positive = len(gold_segments & predicted_segments)
    false_positive = len(predicted_segments - gold_segments)
    false_negative = len(gold_segments - predicted_segments)
    precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0
    recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    print({
        "paper": review["paper"], "page": review["page"],
        "gold_paragraphs": len(gold_segments),
        "predicted_paragraphs": len(predicted_segments),
        "precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3),
    })


## Task 4 — Sentence segmentation

We compared spaCy's trained parser with a rule-based sentencizer on scientific prose. The parser usually handled citations and abbreviations better, but the selected PDF page exposed two parser splits: after `Eq.` before `(9)`, and after `et al.` before a parenthetical citation. A narrow post-processing rule joins only known abbreviations followed by a parenthesis. We keep the repair count in each output so its impact is visible.

In [ ]:
import json
import spacy
from scipaper.sentences import _join_known_false_splits, segment_paragraphs

nlp = spacy.load("en_core_web_sm")
rule_nlp = spacy.blank("en")
rule_nlp.add_pipe("sentencizer")
manual_path = PROJECT_ROOT / "data" / "annotations" / "task4_manual_review.json"
manual_cases = json.loads(manual_path.read_text(encoding="utf-8"))["reviews"]
source_path = PROJECT_ROOT / "outputs" / "paragraphs" / "1406.1078v3_paragraphs.json"
source_doc = json.loads(source_path.read_text(encoding="utf-8"))

for case in manual_cases:
    paragraph = next(
        p for p in source_doc["paragraphs"]
        if p["page"] == case["page"] and p["paragraph"] == case["paragraph"]
    )
    raw = [s.text.strip() for s in nlp(paragraph["text"]).sents if s.text.strip()]
    repaired, repair_count = _join_known_false_splits(raw)
    rule_based = [s.text.strip() for s in rule_nlp(paragraph["text"]).sents if s.text.strip()]
    print({
        "paragraph": case["paragraph"],
        "expected_sentences": case["expected_sentence_count"],
        "naive_split_dot": len([part for part in paragraph["text"].split(".") if part.strip()]),
        "sentencizer": len(rule_based),
        "spaCy_parser_before": len(raw),
        "after_repair": len(repaired),
        "repairs": repair_count,
        "matches_review": len(repaired) == case["expected_sentence_count"],
    })


### Run Task 4 on all paragraph JSON files

The outputs are intermediate files for Tasks 5–9. They retain the Task 3 paragraph metadata and nest sentence IDs/text under each paragraph.

In [ ]:
from pathlib import Path
from scipaper.sentences import process_file

paragraph_dir = PROJECT_ROOT / "outputs" / "paragraphs"
sentence_dir = PROJECT_ROOT / "outputs" / "sentences"
paragraph_jsons = sorted(paragraph_dir.glob("*_paragraphs.json"))
sentence_results = [process_file(path, nlp, sentence_dir) for path in paragraph_jsons]
sentence_results


## Task 5 — Tokenization and linguistic annotations

We compared spaCy’s default tokenizer with a small, general infix rule for parentheses attached to letters. The rule separates inline forms such as `p(xt)` into `p`, `(`, `xt`, `)` while retaining the model’s other English rules. In the current sentence records it changes tokenization in 130 sentences; this is an exploratory normalization choice, not a claim that all resulting tags for mathematical notation are linguistically meaningful.

Each corrected Task 4 sentence is parsed independently. Token indices restart at zero for each sentence. Punctuation is retained, and the whitespace after each token is stored so the original sentence can be reconstructed. Lemma, coarse POS, dependency label, and dependency head come from spaCy.


In [ ]:
import spacy
from scipaper.tokens import configure_scientific_tokenizer, process_file

task5_nlp = spacy.load("en_core_web_sm")
configure_scientific_tokenizer(task5_nlp)
sentence_dir = PROJECT_ROOT / "outputs" / "sentences"
token_dir = PROJECT_ROOT / "outputs" / "tokens"
sentence_jsons = sorted(sentence_dir.glob("*_sentences.json"))
token_results = [process_file(path, task5_nlp, token_dir) for path in sentence_jsons]
token_results


## Task 6 — Part-of-speech statistics

Raw POS counts favor longer papers, so report both counts and percentages. POS percentages use all non-punctuation spaCy tokens as the denominator; the required nine tags are shown separately and all remaining non-punctuation tags are grouped as `OTHER`. Punctuation counts and total token counts are retained separately.

The table below compares the main paper (`AttentionYouNeed`) with `N16-1024` and `1607.06450v1`. Differences are descriptive only: references, inline mathematical symbols, and spaCy tagging errors can affect the distributions.


In [ ]:
import json
import pandas as pd
from scipaper.pos_stats import process_files

token_dir = PROJECT_ROOT / "outputs" / "tokens"
statistics_dir = PROJECT_ROOT / "outputs" / "statistics"
token_jsons = sorted(token_dir.glob("*_tokens.json"))
pos_rows = process_files(token_jsons, statistics_dir)
selected_papers = {"AttentionYouNeed", "N16-1024", "1607.06450v1"}
pos_comparison = pd.DataFrame([
    {"paper": row["paper"], "tokens": row["total_tokens_including_punctuation"],
     **{f"{pos}_count": row["pos_counts"][pos] for pos in ("NOUN", "VERB", "ADJ", "ADV", "PROPN", "PRON", "DET", "ADP", "NUM")},
     **{f"{pos}_pct": row["pos_percentages"][pos] for pos in ("NOUN", "VERB", "ADJ", "ADV", "PROPN", "PRON", "DET", "ADP", "NUM")}}
    for row in pos_rows if row["paper"] in selected_papers
]).set_index("paper")
pos_comparison


## Task 7 — Dependency parsing and visualization

The curated sample contains 10 sentences from 10 distinct sections across the nine papers. The token table includes token, lemma, POS, dependency label, and syntactic head. Three cases are discussed and shown with displaCy below. The parser output is a model prediction; technical notation and PDF extraction artifacts can lead to incorrect attachments. The full sample manifest is `data/annotations/task7_dependency_examples.json`.


In [ ]:
import json
import pandas as pd
import spacy
from IPython.display import IFrame, display
from scipaper.dependencies import build_dependency_examples
from scipaper.tokens import configure_scientific_tokenizer

dependency_nlp = spacy.load("en_core_web_sm")
configure_scientific_tokenizer(dependency_nlp)
dependency_manifest = PROJECT_ROOT / "data" / "annotations" / "task7_dependency_examples.json"
dependency_summary = build_dependency_examples(
    dependency_manifest, PROJECT_ROOT / "outputs" / "tokens",
    PROJECT_ROOT / "outputs" / "dependency_examples", dependency_nlp
)
dependency_summary


In [ ]:
dependency_csv = PROJECT_ROOT / "outputs" / "dependency_examples" / "dependency_tokens.csv"
dependency_table = pd.read_csv(dependency_csv)
display(dependency_table[["case_id", "section", "token", "lemma", "pos", "dependency", "head"]])


### Three dependency structures

1. **Passive training objective:** `trained` is the root; `networks` is its passive subject and `are` its passive auxiliary. `maximize` is an open complement, and `probability` is the object of `maximize`.
2. **Relative clause:** In the machine-reader sentence, `present` is the root. The relative clause `which is designed...` modifies `reader`; `designed` heads the clause and `process` is its complement.
3. **Coordinated measures:** In the results sentence, `report` is the root and `we` is its subject. The measures following it form a coordinated list. spaCy attaches parts of the list imperfectly, so this is a useful example of where the diagram needs human interpretation.

### displaCy diagrams


In [ ]:
diagram_dir = PROJECT_ROOT / "outputs" / "dependency_examples"
for filename in [
    "passive_training_objective.html",
    "machine_reader_relative_clause.html",
    "reporting_experimental_results.html",
]:
    display(IFrame(src=str(diagram_dir / filename), width="100%", height=520))


## Task 8 — Document statistics

The table summarizes all nine papers. Total spaCy tokens include punctuation. Average sentence and paragraph lengths count non-punctuation tokens; unique token types are casefolded surface forms with punctuation excluded. Section counts are detected heading candidates, not manually verified ground truth. Definitions are also recorded in `outputs/statistics/document_statistics.json`.


In [ ]:
import pandas as pd

document_stats_path = PROJECT_ROOT / "outputs" / "statistics" / "document_statistics.csv"
document_stats = pd.read_csv(document_stats_path)
display(document_stats)
